In [13]:
import pandas as pd

def process_payments(input_file='payments_3.csv', output_file='payments.csv'):
    # 1. Đọc dữ liệu thô từ file đầu vào
    try:
        df = pd.read_csv(input_file)
    except FileNotFoundError:
        df = pd.read_csv('payments.csv')

    df_clean = pd.DataFrame()

    # 2. Xử lý chuẩn hóa các cột theo schema PAYMENTS

    # Payment_ID: Tạo mã định danh khóa chính tự động (VD: PAY_0000001)
    if 'Payment_ID' in df.columns:
        df_clean['Payment_ID'] = df['Payment_ID'].astype(str).str.strip()
    else:
        df_clean['Payment_ID'] = ['PAY_' + str(i + 1).zfill(7) for i in range(len(df))]

    # Payment_Date: Chuẩn hóa ngày giờ thanh toán
    if 'Payment_Date' in df.columns:
        df_clean['Payment_Date'] = pd.to_datetime(df['Payment_Date'], errors='coerce')
    else:
        df_clean['Payment_Date'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')

    # Payment_Method: Chuẩn hóa hình thức thanh toán
    method_col = 'payment_method' if 'payment_method' in df.columns else 'Payment_Method'
    if method_col in df.columns:
        df_clean['Payment_Method'] = df[method_col].astype(str).str.strip().str.upper()
    else:
        df_clean['Payment_Method'] = 'UNKNOWN'

    # Amount_Paid: Làm sạch số tiền thanh toán (ép kiểu số, loại âm, làm tròn 2 chữ số)
    val_col = 'payment_value' if 'payment_value' in df.columns else 'Amount_Paid'
    if val_col in df.columns:
        if df[val_col].dtype == object:
            df[val_col] = df[val_col].astype(str).str.replace(r'[^\d.]', '', regex=True)
        df_clean['Amount_Paid'] = pd.to_numeric(df[val_col], errors='coerce').fillna(0.0).clip(lower=0.0).round(2)
    else:
        df_clean['Amount_Paid'] = 0.0

    # Payment_Status: Xác định trạng thái giao dịch
    if 'Payment_Status' in df.columns:
        df_clean['Payment_Status'] = df['Payment_Status'].astype(str).str.strip().str.upper()
    else:
        df_clean['Payment_Status'] = df_clean['Amount_Paid'].apply(lambda x: 'COMPLETED' if x > 0 else 'PENDING')

    # Order_ID: Khóa ngoại tham chiếu bảng đơn hàng
    order_col = 'order_id' if 'order_id' in df.columns else 'Order_ID'
    if order_col in df.columns:
        df_clean['Order_ID'] = df[order_col].astype(str).str.strip()
    else:
        df_clean['Order_ID'] = 'UNKNOWN'

    # 3. Lọc trùng lặp và loại bỏ dòng thiếu khóa chính
    cols = ['Payment_ID', 'Payment_Date', 'Payment_Method', 'Amount_Paid', 'Payment_Status', 'Order_ID']
    payments_table = df_clean[cols].drop_duplicates(subset=['Payment_ID']).dropna(subset=['Payment_ID'])

    # 4. Xuất file CSV
    payments_table.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng PAYMENTS: {output_file} ({len(payments_table):,} dòng)")

if __name__ == '__main__':
    process_payments()

-> Xuất thành công bảng PAYMENTS: payments.csv (646,945 dòng)


In [23]:
# payment
import pandas as pd

def process_payment_methods(input_file='payments_3.csv', output_file='payment_methods.csv'):
    try:
        df = pd.read_csv(input_file)
    except FileNotFoundError:
        df = pd.read_csv('payments.csv')

    method_col = 'payment_method' if 'payment_method' in df.columns else 'Payment_Method'

    # Trích xuất danh mục phương thức duy nhất
    methods = pd.DataFrame(df[method_col].unique(), columns=['method_code'])
    methods['method_code'] = methods['method_code'].astype(str).str.strip().str.lower()
    methods = methods.drop_duplicates().reset_index(drop=True)

    methods['method_id'] = range(1, len(methods) + 1)
    methods['method_name'] = methods['method_code'].str.replace('_', ' ').str.title()
    methods['supports_installments'] = methods['method_code'].isin(['credit_card', 'paypal'])

    cols = ['method_id', 'method_code', 'method_name', 'supports_installments']
    methods_df = methods[cols]

    methods_df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng PAYMENT_METHODS: {output_file} ({len(methods_df):,} dòng)")

if __name__ == '__main__':
    process_payment_methods()

-> Xuất thành công bảng PAYMENT_METHODS: payment_methods.csv (5 dòng)


In [24]:
import pandas as pd

def process_orders_summary(input_file='payments_3.csv', output_file='orders_summary.csv'):
    try:
        df = pd.read_csv(input_file)
    except FileNotFoundError:
        df = pd.read_csv('payments.csv')

    order_col = 'order_id' if 'order_id' in df.columns else 'Order_ID'
    val_col = 'payment_value' if 'payment_value' in df.columns else 'Amount_Paid'
    method_col = 'payment_method' if 'payment_method' in df.columns else 'Payment_Method'

    df['total_value'] = pd.to_numeric(df[val_col], errors='coerce').fillna(0.0).clip(lower=0.0)

    # Tổng hợp giá trị và số lượt thanh toán theo từng Order_ID
    order_df = df.groupby(order_col).agg(
        Total_Amount_Paid=('total_value', 'sum'),
        Payment_Count=(method_col, 'count'),
        Primary_Method=(method_col, 'first')
    ).reset_index()

    order_df.rename(columns={order_col: 'Order_ID'}, inplace=True)
    order_df['Total_Amount_Paid'] = order_df['Total_Amount_Paid'].round(2)

    order_df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng ORDERS_SUMMARY: {output_file} ({len(order_df):,} dòng)")

if __name__ == '__main__':
    process_orders_summary()

-> Xuất thành công bảng ORDERS_SUMMARY: orders_summary.csv (646,945 dòng)
